# Sydney flats: exploratory data analysis

Self-contained: this notebook needs only the three raw CSVs, and it does not import `src/`.

**Privacy.** The data is for use on the day only. Keep the Drive folder private, do not share this
notebook once it has outputs, and keep to aggregates: no `.head()` on raw tables, no addresses in cells you save.

**Run order.** Set the config cell, run the detection cell, check its table, then Run all.

## 0. Config

In [ ]:
import os
from pathlib import Path

DEMO = False                                   # True reads mock CSVs instead of the real files
DRIVE_DIR = "/content/drive/MyDrive/flats_private"   # the private Drive folder
MOCK_DIR = os.environ.get("FLATS_DATA_DIR") or f"{DRIVE_DIR}/mock"
DEMO = DEMO or os.environ.get("FLATS_DEMO") == "1"

INPUT_FILE = "rents-input-flats-city-of-sydney.csv"
LEASES_FILE = "rents-past-leases-city-of-sydney.csv"
BLOCKS_FILE = "blocks-homes-and-people-2021-2026.csv"

AS_OF = "2026-07-31"                           # nothing after this date may be used
FOLD_CUTOFFS = {"as-of": "2026-07-31", "Fold A": "2026-05-31", "Fold B": "2026-03-31"}
FOLD_WINDOWS = {"Fold A": ("2026-06-01", "2026-07-31"), "Fold B": ("2026-04-01", "2026-05-31")}
RENT_LO, RENT_HI = 100, 10_000                 # weekly dollars; same bounds as src/load.py

# Lease column names are unknown. Leave None to auto-detect, or set the real column name.
OVERRIDE = dict(date=None, rent=None, flat_id=None, address=None, building=None,
                parking=None, bedrooms=None, bathrooms=None)

In [ ]:
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Not on Colab: skipping Drive mount.")

DATA = Path(MOCK_DIR if DEMO else DRIVE_DIR)
for name in (INPUT_FILE, LEASES_FILE, BLOCKS_FILE):
    assert (DATA / name).exists(), f"Missing {DATA / name}. Check DRIVE_DIR, or set DEMO."
print("*** MOCK DATA: nothing below describes the real market ***" if DEMO else "Real data.")

AS_OF_TS = pd.Timestamp(AS_OF)
BLUE, ORANGE, GREY = "#2a78d6", "#d9730d", "#8a8986"
plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e6e5e0", "axes.axisbelow": True,
                     "axes.titlesize": 12, "axes.titlelocation": "left", "lines.linewidth": 2})
pd.set_option("display.max_rows", 80, "display.max_columns", 40, "display.float_format", "{:,.2f}".format)

F = pd.read_csv(DATA / INPUT_FILE, dtype={"gnaf_id": "string", "mb": "string"}, low_memory=False)
B = pd.read_csv(DATA / BLOCKS_FILE, dtype={"mb": "string"}, low_memory=False)
RAW = pd.read_csv(DATA / LEASES_FILE, low_memory=False)
print(f"input {F.shape}, leases {RAW.shape}, blocks {B.shape}")

## 0a. Detect lease columns

Check this table before anything else. If a pick is wrong, set it in `OVERRIDE` and rerun from the config cell.
Also check the median rent: a value near 3,000 means the column is monthly, not weekly.

In [ ]:
SENTINEL = "0001-01-01"


def parse_dates(s):
    """Dates as Timestamps. ISO first, then day-first. The 0001-01-01 'unknown' sentinel becomes NaT."""
    t = s.astype("string").str.strip().str.split(r"[T ]", regex=True).str[0]
    t = t.mask(t.eq(SENTINEL))
    d = pd.to_datetime(t, errors="coerce", format="%Y-%m-%d")
    if d.notna().sum() < 0.5 * t.notna().sum():
        d = pd.to_datetime(t, errors="coerce", dayfirst=True, format="mixed")
    return d


def parse_money(s):
    """Numbers from 650, '650', '$1,200 per week'. Takes the first number in the text."""
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    t = s.astype("string").str.replace(",", "", regex=False).str.extract(r"(\d+(?:\.\d+)?)")[0]
    return pd.to_numeric(t, errors="coerce")


def to_flag(s):
    """1.0 / 0.0 / NaN from numbers, booleans, yes/no text."""
    t = s.astype("string").str.strip().str.lower()
    words = t.map({"true": 1.0, "yes": 1.0, "y": 1.0, "false": 0.0, "no": 0.0, "n": 0.0}).astype(float)
    num = pd.to_numeric(t, errors="coerce")
    return words.fillna((num > 0).astype(float).where(num.notna()))


def norm_addr(s):
    return (s.astype("string").str.upper().str.replace(r"[^A-Z0-9 ]", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True).str.strip())


def date_candidates(df):
    rows = []
    for c in df.columns:
        v = df[c].dropna()
        if v.empty or pd.api.types.is_float_dtype(v):
            continue
        v = v.sample(min(5000, len(v)), random_state=0)
        rate = parse_dates(v).notna().mean() + v.astype("string").str.startswith(SENTINEL).mean()
        if rate < 0.8:
            continue
        low = c.lower()
        hint = 2 * bool(re.search(r"advert|list|lease|start|first", low)) + bool(re.search(r"date|_at|_on", low))
        hint -= 3 * bool(re.search(r"end|last|update|modif|sold|expir|creat", low))
        rows.append(dict(column=c, parse_rate=rate, hint=hint))
    return pd.DataFrame(rows, columns=["column", "parse_rate", "hint"]).sort_values(
        ["hint", "parse_rate"], ascending=False, ignore_index=True)


def rent_candidates(df):
    rows = []
    for c in df.columns:
        low = c.lower()
        if re.search(r"(^|_)id$|bond|lat|lon|post|sqm|area|bed|bath|car|floor|addr|street|unit", low):
            continue
        x = parse_money(df[c].dropna().sample(min(5000, df[c].notna().sum()), random_state=0)) if df[c].notna().any() else None
        if x is None or x.notna().mean() < 0.8:
            continue
        med = x.median()
        hint = 3 * bool(re.search(r"rent", low)) + 2 * bool(re.search(r"price|weekly|_pw|asking", low)) + (200 <= med <= 5000)
        if hint:
            rows.append(dict(column=c, median=med, hint=hint))
    return pd.DataFrame(rows, columns=["column", "median", "hint"]).sort_values("hint", ascending=False, ignore_index=True)


def by_name(df, pattern, exclude=None):
    hits = [c for c in df.columns if re.search(pattern, c.lower()) and not (exclude and re.search(exclude, c.lower()))]
    return hits[0] if hits else None


dates, rents = date_candidates(RAW), rent_candidates(RAW)
COLS = dict(
    date=dates.column.iloc[0] if len(dates) else None,
    rent=rents.column.iloc[0] if len(rents) else None,
    flat_id=by_name(RAW, r"gnaf|flat_id|property_id"),
    address=by_name(RAW, r"address", r"building"),
    building=by_name(RAW, r"building"),
    parking=by_name(RAW, r"car|parking|garage"),
    bedrooms=by_name(RAW, r"^bed"),
    bathrooms=by_name(RAW, r"^bath"),
)
COLS.update({k: v for k, v in OVERRIDE.items() if v})
for k, v in COLS.items():
    assert v is None or v in RAW.columns, f"OVERRIDE {k}={v!r} is not a column"

print("All lease columns:", list(RAW.columns))
print("\nDate candidates (best first):"); display(dates)
print("Rent candidates (best first):"); display(rents)
print("Using:"); display(pd.Series(COLS, name="lease column").to_frame())
assert COLS["date"] and COLS["rent"], "Could not detect date or rent. Set them in OVERRIDE."

## 0b. Standardise, then filter to dates <= as-of

Rows after the as-of date are counted, then dropped here, so no later cell can see them.
Their rents are never read. Everything below works from `L`, or `Lc` once rent outliers are removed.

In [ ]:
F["building_key"] = norm_addr(F["building_address"])
F["attrs_last_seen_d"] = parse_dates(F["attrs_last_seen"])
F["first_ever_listed_d"] = parse_dates(F["first_ever_listed"])
F["ever_adv"] = to_flag(F["ever_advertised_for_lease"])
input_ids = set(F.gnaf_id.dropna())

# Link each lease to an input flat: by id if the id overlaps the input, otherwise by address.
link = pd.Series(pd.NA, index=RAW.index, dtype="string")
link_method = "none"
if COLS["flat_id"] and RAW[COLS["flat_id"]].astype("string").isin(input_ids).mean() > 0.01:
    link, link_method = RAW[COLS["flat_id"]].astype("string"), "id"
elif COLS["address"]:
    k = norm_addr(F["address"])
    keep = ~k.duplicated()
    link = norm_addr(RAW[COLS["address"]]).map(dict(zip(k[keep], F.gnaf_id[keep]))).astype("string")
    link_method = "address"
print("Link method:", link_method)

raw_date = parse_dates(RAW[COLS["date"]])
raw_text = RAW[COLS["date"]].astype("string").str.strip()
late = raw_date > AS_OF_TS
Q = dict(rows=len(RAW), sentinel=int(raw_text.str.startswith(SENTINEL).sum()), blank=int(raw_text.isna().sum()),
         unparsed=int((raw_date.isna() & raw_text.notna() & ~raw_text.str.startswith(SENTINEL, na=False)).sum()),
         after_as_of=int(late.sum()), last_kept=raw_date[~late].max())
early_ids, late_ids = set(link[~late & raw_date.notna()].dropna()), set(link[late].dropna())
Q["flagged_flats_only_late_leases"] = int(F.gnaf_id.isin(late_ids - early_ids)[F.ever_adv == 1].sum())

keep = raw_date.notna() & ~late
L_RAW = RAW[keep]                   # original columns, for the schema table
L = pd.DataFrame({"date": raw_date[keep], "gnaf_id": link[keep], "rent": parse_money(RAW.loc[keep, COLS["rent"]])})
for std, col, fn in (("advert_park", "parking", to_flag), ("beds_lease", "bedrooms", parse_money), ("baths_lease", "bathrooms", parse_money)):
    L[std] = fn(RAW.loc[keep, COLS[col]]) if COLS[col] else np.nan
L["building_lease"] = norm_addr(RAW.loc[keep, COLS["building"]]) if COLS["building"] else pd.NA
del RAW, raw_date, raw_text, link, late, late_ids, early_ids, keep

attrs = F[["gnaf_id", "building_key", "suburb", "bedrooms", "bathrooms", "car_space", "floor_level", "property_type"]].rename(
    columns={"bedrooms": "beds_input", "bathrooms": "baths_input", "car_space": "input_park"})
attrs["input_park"] = to_flag(attrs["input_park"])
L = L.merge(attrs, on="gnaf_id", how="left")
L["building_key"] = L["building_lease"].astype("string").fillna(L["building_key"])
if L.suburb.isna().any():
    L["suburb"] = L["suburb"].fillna(L.building_key.map(F.drop_duplicates("building_key").set_index("building_key").suburb))
L["beds"] = L.beds_lease.fillna(L.beds_input)
L["baths"] = L.baths_lease.fillna(L.baths_input)
L["park"] = L.advert_park.fillna(L.input_park)
L["month"] = L.date.dt.to_period("M").dt.to_timestamp()
L["linked"] = L.gnaf_id.isin(input_ids)

assert L.date.max() <= AS_OF_TS, "a row after the as-of date survived"
Lc = L[L.rent.between(RENT_LO, RENT_HI)].copy()
print(f"Kept {len(L):,} of {Q['rows']:,} lease rows; {len(Lc):,} with rent in range. Last date kept: {L.date.max():%Y-%m-%d}")
print(f"Linked to an input flat: {L.linked.mean():.1%}")

**Shared helpers.** `rent_adj` restates each rent at as-of prices, using a monthly index with suburb and
bedroom mix removed. It sees all history up to as-of, so it is for exploring only. Do not reuse it as a fold
feature. `pairs` holds consecutive leases of the same flat.

In [ ]:
Lc["log_rent"] = np.log(Lc.rent)
cell_mean = Lc.groupby(["suburb", "beds"], dropna=False).log_rent.transform("mean")
idx = (Lc.log_rent - cell_mean).groupby(Lc.month).median()
idx = idx.reindex(pd.date_range(idx.index.min(), idx.index.max(), freq="MS")).interpolate()
idx = idx.rolling(3, center=True, min_periods=1).mean()
Lc["rent_adj"] = np.exp(Lc.log_rent - Lc.month.map(idx) + idx.iloc[-1])

pairs = Lc[Lc.linked].sort_values(["gnaf_id", "date"])
g = pairs.groupby("gnaf_id")
pairs = pairs.assign(prev_rent=g.rent.shift(), prev_date=g.date.shift(), prev_month=g.month.shift(), prev_park=g.park.shift())
pairs = pairs[pairs.prev_rent.notna()].copy()
pairs["gap_days"] = (pairs.date - pairs.prev_date).dt.days
pairs["indexed_prev"] = pairs.prev_rent * np.exp(pairs.month.map(idx) - pairs.prev_month.map(idx))


def plot_or_note(df, msg="Not enough data for this view."):
    if df is None or len(df) == 0:
        print(msg)
        return False
    return True

## 1. Schema and nulls

**Look for:** columns nobody mentioned, null rates, ids that are not unique, and how complete `floor_level` is.

**Informs:** the `LEASE_ALIASES` map in `src/load.py`, which columns can be features, and whether a missing-value flag is needed.

In [ ]:
def schema(df):
    return pd.DataFrame({"dtype": df.dtypes.astype(str), "nulls": df.isna().sum(), "null_pct": 100 * df.isna().mean(),
                         "distinct": df.nunique()})


for name, df in (("Input flats", F.drop(columns=["building_key", "attrs_last_seen_d", "first_ever_listed_d", "ever_adv"])),
                 ("Leases (<= as-of)", L_RAW), ("Blocks", B)):
    print(f"{name}: {len(df):,} rows"); display(schema(df))
print("Input gnaf_id unique:", F.gnaf_id.is_unique)
print("Input mesh blocks found in blocks file:", f"{F.mb.isin(B.mb).mean():.1%}")

## 2. Rent distribution and outliers

**Look for:** a weekly-versus-monthly mix (a second hump near 4x), round-number heaping, and values out of range.

**Informs:** the `RENT_MIN` and `RENT_MAX` bounds, a log target, and L1 over L2. Scoring is median absolute error, so the bulk matters more than the tail.

In [ ]:
print("Rent quantiles, all kept rows:")
display(L.rent.quantile([0, .001, .01, .05, .25, .5, .75, .95, .99, .999, 1]).to_frame("rent"))
print(f"Below {RENT_LO}: {(L.rent < RENT_LO).sum():,}; above {RENT_HI}: {(L.rent > RENT_HI).sum():,}; unparsed: {L.rent.isna().sum():,}")
print(f"Rents divisible by 5: {(Lc.rent % 5 == 0).mean():.1%}; by 50: {(Lc.rent % 50 == 0).mean():.1%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].hist(Lc.rent.clip(upper=Lc.rent.quantile(.995)), bins=80, color=BLUE)
ax[0].set(title="Weekly rent (top 0.5% clipped)", xlabel="$ per week", ylabel="leases")
ax[1].hist(Lc.log_rent, bins=80, color=BLUE)
ax[1].set(title="Log weekly rent", xlabel="log($ per week)", ylabel="leases")
plt.tight_layout(); plt.show()

## 3. Monthly median rent and the time-lag effect

**Look for:** the growth rate in the last 12 to 24 months, thin early months, and how fast an old rent goes stale.

**Informs:** index adjustment of anchors, how much weight an old anchor deserves, and the two-month forward step.
If the indexed error stays flat as the gap grows, old anchors remain useful.

In [ ]:
m = Lc.groupby("month").rent.agg(["median", "size"])
fig, ax = plt.subplots(2, 1, figsize=(10, 5.5), sharex=True, height_ratios=[3, 1])
ax[0].plot(m.index, m["median"], color=GREY, label="raw median")
ax[0].plot(idx.index, np.exp(idx - idx.iloc[-1]) * Lc[Lc.month >= Lc.month.max() - pd.DateOffset(months=2)].rent.median(),
           color=BLUE, label="mix-adjusted index")
ax[0].set(title="Monthly rent", ylabel="$ per week"); ax[0].legend(frameon=False)
ax[1].bar(m.index, m["size"], width=25, color=GREY); ax[1].set(ylabel="leases")
plt.tight_layout(); plt.show()

i = np.exp(idx)
print("Mix-adjusted growth over the previous 12 months:")
display((100 * (i / i.shift(12) - 1)).iloc[-25::6].to_frame("% growth"))

bins = [0, 90, 180, 365, 730, 1095, 1825, 100_000]
labels = ["<3m", "3-6m", "6-12m", "1-2y", "2-3y", "3-5y", "5y+"]
pairs["gap_bucket"] = pd.cut(pairs.gap_days, bins, labels=labels)
lag = pairs.groupby("gap_bucket", observed=True).apply(lambda d: pd.Series({
    "pairs": len(d), "median_abs_err_raw": (d.rent - d.prev_rent).abs().median(),
    "median_abs_err_indexed": (d.rent - d.indexed_prev).abs().median(),
    "median_change_pct": 100 * (d.rent / d.prev_rent - 1).median()}), include_groups=False)
print("Using the flat's previous rent to predict its next rent:"); display(lag)
if plot_or_note(lag):
    plt.plot(lag.index.astype(str), lag.median_abs_err_raw, marker="o", color=GREY, label="previous rent as is")
    plt.plot(lag.index.astype(str), lag.median_abs_err_indexed, marker="o", color=BLUE, label="previous rent, indexed")
    plt.title("Error of a stale rent, by time since that lease"); plt.ylabel("median abs error, $"); plt.xlabel("gap")
    plt.legend(frameon=False); plt.show()

## 4. Rent by suburb and bedrooms

**Look for:** thin suburb and bedroom cells, and suburbs whose spread is much wider than others.

**Informs:** the fallback levels in the hierarchical median, `MIN_GROUP`, and how to encode suburb. Recent means the last 12 months.

In [ ]:
recent = Lc[Lc.date > AS_OF_TS - pd.DateOffset(months=12)]
beds_c = recent.assign(beds=recent.beds.clip(upper=4))
print("Median rent, last 12 months (4 means 4+):")
display(beds_c.pivot_table(index="suburb", columns="beds", values="rent", aggfunc="median"))
print("Lease counts:")
cnt = beds_c.pivot_table(index="suburb", columns="beds", values="rent", aggfunc="size")
display(cnt)
print(f"Cells under 5 leases: {(cnt.fillna(0) < 5).to_numpy().mean():.1%}")

s = recent.groupby("suburb").rent.agg(median="median", q25=lambda x: x.quantile(.25), q75=lambda x: x.quantile(.75), n="size")
s = s[s.n >= 20].sort_values("median")
if plot_or_note(s):
    fig, ax = plt.subplots(figsize=(9, max(3, .26 * len(s))))
    ax.hlines(s.index, s.q25, s.q75, color=GREY, lw=2)
    ax.plot(s["median"], s.index, "o", color=BLUE, ms=6)
    ax.set(title="Rent by suburb, last 12 months (dot: median, bar: middle half)", xlabel="$ per week")
    plt.tight_layout(); plt.show()

## 5. With and without parking

**Look for:** the premium within suburb and bedrooms, then within building. The building figure is the cleaner one,
because parking goes with larger, newer flats.

**Informs:** the size of the gap between `rent_with_car_space` and `rent_without_car_space`, and whether it varies by suburb.

In [ ]:
def premium(d, keys, min_n=10):
    t = d.dropna(subset=["park"] + keys).groupby(keys + ["park"]).rent_adj.agg(["median", "size"]).unstack("park")
    if t.shape[1] < 4:
        return pd.DataFrame(columns=["premium", "n"])
    t = t[(t[("size", 0.0)] >= min_n) & (t[("size", 1.0)] >= min_n)]
    return pd.DataFrame({"premium": t[("median", 1.0)] - t[("median", 0.0)], "n": t["size"].sum(axis=1)})


print(f"Parking known: {Lc.park.notna().mean():.1%}; of those with parking: {Lc.park.mean():.1%}")
p_sb = premium(Lc, ["suburb", "beds"])
p_bb = premium(Lc, ["building_key", "beds"], min_n=3)
for name, p in (("suburb x bedrooms", p_sb), ("building x bedrooms", p_bb)):
    print(f"{name}: median premium ${p.premium.median():,.0f} per week across {len(p):,} cells")
if len(p_sb):
    print("Premium by bedrooms:")
    display(p_sb.reset_index().groupby("beds").premium.describe()[["count", "25%", "50%", "75%"]])
    by_sub = p_sb.reset_index().groupby("suburb").premium.median().sort_values()
    fig, ax = plt.subplots(figsize=(9, max(3, .26 * len(by_sub))))
    ax.barh(by_sub.index, by_sub.values, color=BLUE, height=.7)
    ax.set(title="Median parking premium by suburb, at as-of prices", xlabel="$ per week")
    plt.tight_layout(); plt.show()

## 6. Floor level against rent, within building

**Look for:** the slope once building and bedrooms are held fixed, and whether it bends at higher floors.
Comparing across buildings would mostly measure which buildings are tall.

**Informs:** the floor curve in the parking lane, and how to treat flats with no `floor_level`.

In [ ]:
print(f"Input flats with floor_level: {F.floor_level.notna().mean():.1%}; leases: {Lc.floor_level.notna().mean():.1%}")
w = Lc.dropna(subset=["floor_level", "beds", "building_key"]).copy()
grp = w.groupby(["building_key", "beds"])
w = w[(grp.rent_adj.transform("size") >= 5) & (grp.floor_level.transform("nunique") >= 3)]
if plot_or_note(w):
    grp = w.groupby(["building_key", "beds"])
    w["d_log"] = np.log(w.rent_adj) - np.log(w.rent_adj).groupby([w.building_key, w.beds]).transform("mean")
    w["d_floor"] = w.floor_level - grp.floor_level.transform("mean")
    slope = np.polyfit(w.d_floor, w.d_log, 1)[0]
    print(f"{len(w):,} leases in {grp.ngroups:,} building-bedroom groups")
    print(f"Slope: {100 * slope:.2f}% per floor, about ${slope * w.rent_adj.median():.1f} per week at the median rent")
    b = w.groupby(pd.cut(w.d_floor, [-60, -15, -10, -5, -2, 2, 5, 10, 15, 60]), observed=True).agg(
        floors=("d_floor", "mean"), rent_pct=("d_log", lambda x: 100 * x.median()), n=("d_log", "size"))
    display(b)
    plt.plot(b.floors, b.rent_pct, marker="o", color=BLUE)
    plt.title("Rent against floor, relative to the building average")
    plt.xlabel("floors above the building's average"); plt.ylabel("rent difference, %"); plt.show()

## 7. Repeat leases per flat and time between them

**Look for:** how many flats have several leases, and very short gaps. A gap of days is probably a re-listing of the
same vacancy, not a new lease.

**Informs:** de-duplication, the own-flat anchor, and whether fold test rows should count one flat twice.

In [ ]:
per_flat = Lc[Lc.linked].groupby("gnaf_id").size()
print("Leases per flat that has any:")
display(per_flat.clip(upper=10).value_counts().sort_index().to_frame("flats"))
print(f"Gap quantiles (days):"); display(pairs.gap_days.quantile([.01, .05, .25, .5, .75, .95]).to_frame("days"))
print(f"Same day: {(pairs.gap_days == 0).sum():,}; under 30 days: {(pairs.gap_days < 30).sum():,}; of {len(pairs):,} pairs")
if plot_or_note(pairs):
    plt.hist(pairs.gap_days.clip(upper=2000), bins=80, color=BLUE)
    plt.title("Days between consecutive leases of a flat (capped at 2,000)"); plt.xlabel("days"); plt.ylabel("pairs"); plt.show()

## 8. Coverage of input flats by own and building history

**Look for:** the share of flats with no own lease, and whether scored adverts look like all input flats. Flats that
get advertised are likelier to have history, so all-flat coverage understates it.

**Informs:** how much rests on `anchor_own`, `anchor_bldg` and `anchor_block`, and which fallback matters most.

In [ ]:
def coverage(ids, bkeys, cutoff):
    h = Lc[Lc.date <= cutoff]
    own = ids.isin(h.gnaf_id.dropna())
    own24 = ids.isin(h[h.date > cutoff - pd.DateOffset(months=24)].gnaf_id.dropna())
    bld = bkeys.isin(h.building_key.dropna())
    bld12 = bkeys.isin(h[h.date > cutoff - pd.DateOffset(months=12)].building_key.dropna())
    return {"rows": len(ids), "own_any_%": 100 * own.mean(), "own_24m_%": 100 * own24.mean(), "building_any_%": 100 * bld.mean(),
            "building_12m_%": 100 * bld12.mean(), "building_only_%": 100 * (bld & ~own).mean(), "neither_%": 100 * (~bld & ~own).mean()}


rows = {f"All input flats at {k}": coverage(F.gnaf_id, F.building_key, pd.Timestamp(v)) for k, v in FOLD_CUTOFFS.items()}
for k, (a, b) in FOLD_WINDOWS.items():
    t = Lc[Lc.linked & Lc.date.between(a, b)]
    rows[f"{k} test adverts"] = coverage(t.gnaf_id, t.building_key, pd.Timestamp(FOLD_CUTOFFS[k]))
display(pd.DataFrame(rows).T)

## 9. Advert parking against input parking

**Look for:** how often they disagree, and which way. Scoring uses advert parking first, then input parking, so a
wrong input value costs a parking premium on that row.

**Informs:** `car_space_resolved`, and whether the two-pass prediction needs care for flats where input parking is blank.

In [ ]:
print(f"Advert parking blank: {L.advert_park.isna().mean():.1%}; input parking blank: {F.car_space.isna().mean():.1%}")
both = Lc.dropna(subset=["advert_park", "input_park"])
if plot_or_note(both, "No rows with both advert and input parking."):
    display(pd.crosstab(both.input_park, both.advert_park, normalize="all").mul(100).rename_axis(index="input", columns="advert"))
    print(f"Disagreement: {(both.input_park != both.advert_park).mean():.1%} of {len(both):,} leases")
    print("Median rent at as-of prices, by input and advert parking:")
    display(both.pivot_table(index="input_park", columns="advert_park", values="rent_adj", aggfunc="median"))
    flips = pairs.dropna(subset=["park", "prev_park"])
    print(f"Consecutive leases of a flat where parking changed: {(flips.park != flips.prev_park).mean():.1%}")

## 10. Date quirks

**Look for:** the 0001-01-01 sentinel, rows after the as-of date (flagged and dropped in 0b), very old dates, and
heaping on one weekday or the first of the month.

**Informs:** `parse_date`, the as-of filter, and how far back training can go.

In [ ]:
display(pd.Series({"lease rows in file": Q["rows"], "0001-01-01 sentinel": Q["sentinel"], "blank": Q["blank"],
                   "unparsed": Q["unparsed"], "after as-of, dropped": Q["after_as_of"]}, name="count").to_frame())
print(f"Last lease date kept: {Q['last_kept']:%Y-%m-%d}")
rows = {}
for c, d in (("attrs_last_seen", F.attrs_last_seen_d), ("first_ever_listed", F.first_ever_listed_d)):
    t = F[c].astype("string")
    rows[c] = {"blank": int(t.isna().sum()), "sentinel": int(t.str.startswith(SENTINEL, na=False).sum()),
               "unparsed": int((d.isna() & t.notna() & ~t.str.startswith(SENTINEL, na=False)).sum()),
               "after_as_of": int((d > AS_OF_TS).sum()), "min": d.min(), "max": d.max()}
display(pd.DataFrame(rows).T)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
y = L.date.dt.year.value_counts().sort_index()
ax[0].bar(y.index, y.values, color=BLUE); ax[0].set(title="Leases by year")
wd = L.date.dt.day_of_week.value_counts().reindex(range(7), fill_value=0)
ax[1].bar(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], wd.values, color=BLUE); ax[1].set(title="By weekday")
dm = L.date.dt.day.value_counts().sort_index()
ax[2].bar(dm.index, dm.values, color=BLUE); ax[2].set(title="By day of month")
plt.tight_layout(); plt.show()

## 11. Studios with bedrooms > 0

**Look for:** flats typed as studio with a bedroom count, and how their rents compare with true studios and one-beds.

**Informs:** whether to recode them to 0 bedrooms, or keep `property_type` as a separate feature.

In [ ]:
print("Input property types:"); display(F.property_type.value_counts(dropna=False).to_frame("flats"))
studio = F.property_type.astype("string").str.contains("studio", case=False, na=False)
print("Bedrooms among flats typed as studio:")
display(F.loc[studio, "bedrooms"].value_counts(dropna=False).sort_index().to_frame("flats"))
print(f"Typed studio with bedrooms > 0: {(studio & (F.bedrooms > 0)).sum():,}; bedrooms = 0 but not typed studio: {(~studio & (F.bedrooms == 0)).sum():,}")
Lc["studio_type"] = Lc.property_type.astype("string").str.contains("studio", case=False, na=False)
print("Median rent at as-of prices:")
display(Lc[Lc.linked & (Lc.beds <= 2)].pivot_table(index="studio_type", columns="beds", values="rent_adj", aggfunc=["median", "size"]))

## 12. Bedrooms outliers

**Look for:** impossible counts, bedrooms far above bathrooms, and adverts that disagree with the input.

**Informs:** clipping bedrooms, and whether the lease value or the input value should win. `src/load.py` currently lets the lease win.

In [ ]:
print("Input bedrooms:"); display(F.bedrooms.value_counts(dropna=False).sort_index().to_frame("flats"))
print("Median rent at as-of prices by bedrooms:")
display(Lc.groupby("beds").rent_adj.agg(["median", "size"]))
print(f"Input flats with 6+ bedrooms: {(F.bedrooms >= 6).sum():,}; bedrooms at least 3 above bathrooms: {(F.bedrooms - F.bathrooms >= 3).sum():,}")
both = L.dropna(subset=["beds_lease", "beds_input"])
if plot_or_note(both, "Lease file has no bedrooms column, or no linked rows."):
    print(f"Lease bedrooms differ from input: {(both.beds_lease != both.beds_input).mean():.1%} of {len(both):,}")

## 13. Leak checks: `ever_advertised_for_lease` and `attrs_last_seen`

**Look for:** input columns that know about the future. If the flag is 1 for flats with no lease by a cutoff, it was
built from later data. If `attrs_last_seen` falls after a cutoff far more often for flats advertised in the test
window than for other flats, the date itself gives the advert away.

**Informs:** decision 5 in `DECISIONS.md`. Both columns must be rebuilt or masked as of each cutoff, including the final fit.

In [ ]:
print(f"Flag = 1 but the flat's only leases are after as-of: {Q['flagged_flats_only_late_leases']:,} flats")
rows = {}
for k, v in FOLD_CUTOFFS.items():
    c = pd.Timestamp(v)
    known = F.gnaf_id.isin(L[L.date <= c].gnaf_id.dropna())
    rows[k] = {"flag_1": int((F.ever_adv == 1).sum()), "lease_by_cutoff": int(known.sum()),
               "flag_1_no_lease_by_cutoff": int(((F.ever_adv == 1) & ~known).sum()),
               "flag_0_with_lease_by_cutoff": int(((F.ever_adv == 0) & known).sum())}
display(pd.DataFrame(rows).T)

rows = {}
for k, (a, b) in FOLD_WINDOWS.items():
    c = pd.Timestamp(FOLD_CUTOFFS[k])
    t = L[L.linked & L.date.between(a, b)]
    in_test = F.gnaf_id.isin(t.gnaf_id)
    first_time = in_test & ~F.gnaf_id.isin(L[L.date <= c].gnaf_id.dropna())
    rows[k] = {"flats_in_test": int(in_test.sum()), "first_time_advertised": int(first_time.sum()),
               "first_time_with_flag_1_%": 100 * (F.ever_adv[first_time] == 1).mean() if first_time.any() else np.nan,
               "attrs_after_cutoff_test_%": 100 * (F.attrs_last_seen_d[in_test] > c).mean(),
               "attrs_after_cutoff_other_%": 100 * (F.attrs_last_seen_d[~in_test] > c).mean(),
               "listed_after_cutoff_test_%": 100 * (F.first_ever_listed_d[in_test] > c).mean(),
               "listed_after_cutoff_other_%": 100 * (F.first_ever_listed_d[~in_test] > c).mean()}
display(pd.DataFrame(rows).T)

last = L[L.linked].groupby("gnaf_id").date.max().rename("last_lease")
d = F.merge(last, left_on="gnaf_id", right_index=True)
gap = (d.attrs_last_seen_d - d.last_lease).dt.days.dropna()
print("attrs_last_seen minus the flat's last lease date. A spike at 0 means it updates when a flat is advertised:")
display(gap.quantile([.05, .25, .5, .75, .95]).to_frame("days"))
print(f"Within 14 days of the last lease: {(gap.abs() <= 14).mean():.1%}")
if plot_or_note(gap):
    plt.hist(gap.clip(-1500, 1500), bins=80, color=BLUE)
    plt.title("attrs_last_seen minus last lease date (capped at 1,500 days)"); plt.xlabel("days"); plt.ylabel("flats"); plt.show()

## 14. Free exploration

Tables ready to use: `L` (leases up to as-of), `Lc` (rent in range, with `rent_adj`), `pairs` (consecutive leases
of a flat), `F` (input flats), `B` (blocks), `idx` (monthly log index).

Keep to aggregates, and clear outputs before saving a copy anywhere.

In [ ]:
Lc.groupby("beds").rent.median()